# Essay 1 Notebook: Word Embeddings (Lab-Aligned)

This notebook documents how **word/text embeddings** are used in our K-drama lab implementation.

It supports **Essay 1 (Word Embeddings)** with practical evidence from the project pipeline:
- text representation
- embedding generation
- vector-space similarity
- interpretation of results and limitations

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_PATH = ROOT / "data/processed/kdramas_clean.jsonl"
EMB_PATH = ROOT / "data/processed/kdramas_embeddings.jsonl"

print("ROOT:", ROOT)
print("clean exists:", CLEAN_PATH.exists())
print("embeddings exists:", EMB_PATH.exists())

ROOT: /Volumes/SSD/dev/kdrama-semantic-search-recommendation
clean exists: True
embeddings exists: True


In [2]:
def read_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows

clean_rows = read_jsonl(CLEAN_PATH)
emb_rows = read_jsonl(EMB_PATH)

clean_df = pd.DataFrame(clean_rows)
emb_df = pd.DataFrame({
    "imdb_id": [r["imdb_id"] for r in emb_rows],
    "vector": [np.array(r["vector"], dtype=np.float32) for r in emb_rows],
})

df = clean_df.merge(emb_df, on="imdb_id", how="inner")
print(f"clean rows: {len(clean_df)}")
print(f"embedding rows: {len(emb_df)}")
print(f"joined rows: {len(df)}")

df[["imdb_id", "primary_title", "type", "genres"]].head()

clean rows: 28
embedding rows: 28
joined rows: 28


,imdb_id,primary_title,type,genres
0,tt0086817,Transformers,tvSeries,"[Animation, Action, Adventure, Family, Sci-Fi]"
1,tt0077008,Fantasy Island,tvSeries,"[Adventure, Family, Fantasy, Romance]"
2,tt0088510,Star Wars: Droids,tvSeries,"[Animation, Action, Adventure, Comedy, Family,..."
3,tt0081933,The Smurfs,tvSeries,"[Animation, Adventure, Comedy, Family, Fantasy]"
4,tt0088528,Gummi Bears,tvSeries,"[Animation, Adventure, Family, Fantasy]"


In [3]:
# Vector dimension and norm inspection
vec_dim = int(df["vector"].iloc[0].shape[0]) if len(df) else 0
norms = np.array([np.linalg.norm(v) for v in df["vector"]]) if len(df) else np.array([])

print("vector dimension:", vec_dim)
if len(norms):
    print("norm stats -> min/mean/max:", norms.min(), norms.mean(), norms.max())

# For normalized embeddings, norms should be close to 1.0
pd.Series(norms).describe() if len(norms) else None

vector dimension: 384
norm stats -> min/mean/max: 0.99999994 1.0 1.0000001


count    2.800000e+01
mean     1.000000e+00
std      3.215367e-08
min      9.999999e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [4]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def top_k_similar(title: str, k: int = 5):
    idx = df.index[df["primary_title"].str.lower() == title.lower()]
    if len(idx) == 0:
        raise ValueError(f"title not found: {title}")
    i = int(idx[0])
    q = df.loc[i, "vector"]

    sims = []
    for j, row in df.iterrows():
        if j == i:
            continue
        sims.append((row["imdb_id"], row["primary_title"], cosine_sim(q, row["vector"])))
    sims.sort(key=lambda x: x[2], reverse=True)
    return pd.DataFrame(sims[:k], columns=["imdb_id", "title", "cosine_similarity"])

top_k_similar("Rugrats", k=5)

,imdb_id,title,cosine_similarity
0,tt0167743,The Wild Thornberrys,0.454803
1,tt0131664,The Angry Beavers,0.445205
2,tt0105941,Animaniacs,0.442254
3,tt0098929,Tiny Toon Adventures,0.441514
4,tt0118360,Johnny Bravo,0.403083


## Why "Rugrats" may not appear in `top_k_similar("Rugrats")`

`top_k_similar(title)` in this notebook is currently **item-to-item** retrieval and excludes the anchor item itself (`if j == i: continue`).

So if you query the anchor title "Rugrats", the exact same item is intentionally removed from candidate results.

By contrast, the web API (`/api/recommend`) is **query-to-item** retrieval:
- text query -> query embedding -> nearest items in Weaviate

In that setup, "Rugrats" can appear naturally as top result for query text "Rugrats".

In [5]:
def top_k_similar(title: str, k: int = 5, include_self: bool = False):
    idx = df.index[df["primary_title"].str.lower() == title.lower()]
    if len(idx) == 0:
        raise ValueError(f"title not found: {title}")
    i = int(idx[0])
    q = df.loc[i, "vector"]

    sims = []
    for j, row in df.iterrows():
        if (not include_self) and j == i:
            continue
        sims.append((row["imdb_id"], row["primary_title"], cosine_sim(q, row["vector"])))

    sims.sort(key=lambda x: x[2], reverse=True)
    return pd.DataFrame(sims[:k], columns=["imdb_id", "title", "cosine_similarity"])

print("Item-to-item (exclude self):")
display(top_k_similar("Rugrats", k=5, include_self=False))

print("Item-to-item (include self):")
display(top_k_similar("Rugrats", k=5, include_self=True))

Item-to-item (exclude self):


,imdb_id,title,cosine_similarity
0,tt0167743,The Wild Thornberrys,0.454803
1,tt0131664,The Angry Beavers,0.445205
2,tt0105941,Animaniacs,0.442254
3,tt0098929,Tiny Toon Adventures,0.441514
4,tt0118360,Johnny Bravo,0.403083


Item-to-item (include self):


,imdb_id,title,cosine_similarity
0,tt0101188,Rugrats,1.000000
1,tt0167743,The Wild Thornberrys,0.454803
2,tt0131664,The Angry Beavers,0.445205
3,tt0105941,Animaniacs,0.442254
4,tt0098929,Tiny Toon Adventures,0.441514


In [6]:
# Query-to-item example (closer to API behavior)
# This uses the same sentence-transformers model directly in notebook.
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

def query_to_items(query: str, k: int = 5):
    qv = model.encode([query], normalize_embeddings=True)[0]
    sims = []
    for _, row in df.iterrows():
        sims.append((row["imdb_id"], row["primary_title"], cosine_sim(qv, row["vector"])))
    sims.sort(key=lambda x: x[2], reverse=True)
    return pd.DataFrame(sims[:k], columns=["imdb_id", "title", "cosine_similarity"])

query_to_items("Rugrats", k=5)

/Volumes/SSD/dev/kdrama-semantic-search-recommendation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7505.18it/s]


,imdb_id,title,cosine_similarity
0,tt0101188,Rugrats,0.630673
1,tt0081933,The Smurfs,0.283277
2,tt0088631,Thundercats,0.209087
3,tt0131664,The Angry Beavers,0.198860
4,tt0131613,Teenage Mutant Ninja Turtles,0.195194


## Essay 1 Notes

- **Core concepts:** dense vector representation, semantic neighborhood
- **Mathematical foundation:** cosine similarity equation and interpretation
- **Practical applications:** semantic retrieval for K-drama recommendations
- **Advantages:** captures semantic similarity beyond exact keyword overlap
- **Limitations:** small dataset subset, noise in source labels, model/domain mismatch